① データを受け取る

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

# 読み込み元（データ置き場）
data_path = "/content/drive/MyDrive/Kaggle研究会/playground-series-s6e1"

train = pd.read_csv(f"{data_path}/train.csv")
test = pd.read_csv(f"{data_path}/test.csv")
sample_sub = pd.read_csv(f"{data_path}/sample_submission.csv")

print(train.shape, test.shape, sample_sub.shape)

Mounted at /content/drive
(630000, 13) (270000, 12) (270000, 2)


② 数値とカテゴリを分ける
*   目的変数 exam_score を y、id は除外して X を作る → 数値列/カテゴリ列を分ける

In [ ]:
# 元の特徴量だけ（追加特徴量なし）
X = train.drop(["exam_score", "id"], axis=1)
y = train["exam_score"]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("数値特徴量:", list(numeric_features))
print("カテゴリ特徴量:", list(categorical_features))

数値特徴量: ['age', 'study_hours', 'class_attendance', 'sleep_hours']
カテゴリ特徴量: ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']


③ カテゴリを数値に変換（One-Hot）
* カテゴリ列をOne-Hot化（未知カテゴリは無視）

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

④ LightGBMの回帰モデルの定義
* 学習回数や学習率などのハイパーパラメータを設定
* 過学習を抑えるためのパラメータも設定

In [ ]:
!pip -q install lightgbm

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

lgbm = LGBMRegressor(
    n_estimators=2000,      # 早めに止めるので多めでOK
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", lgbm)
    ]
)

⑤ モデルで学習（学習/検証に分割してfit）
* 学習用/検証用に分け、early stopping で過学習を抑えつつ学習

In [ ]:
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

# 学習用/検証用に分割
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ★ 前処理を学習データでfitして、両方を変換
X_train_t = preprocessor.fit_transform(X_train)
X_valid_t = preprocessor.transform(X_valid)

# LightGBM（元のパラメータでOK）
lgbm = LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# ★ eval_set も「変換後」を渡す
lgbm.fit(
    X_train_t, y_train,
    eval_set=[(X_valid_t, y_valid)],
    eval_metric="rmse",
    callbacks=[early_stopping(50), log_evaluation(50)]
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.049686 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 30
[LightGBM] [Info] Start training from score 62.482335
Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 9.38953	valid_0's l2: 88.1633
[100]	valid_0's rmse: 8.84427	valid_0's l2: 78.2211
[150]	valid_0's rmse: 8.80273	valid_0's l2: 77.488
[200]	valid_0's rmse: 8.79185	valid_0's l2: 77.2966
[250]	valid_0's rmse: 8.78604	valid_0's l2: 77.1945
[300]	valid_0's rmse: 8.78166	valid_0's l2: 77.1175
[350]	valid_0's rmse: 8.77737	valid_0's l2: 77.0423
[400]	valid_0's rmse: 8.77433	valid_0's l2: 76.9888
[450]	valid_0's rmse: 8.77231	valid_0's l2: 76.9534
[500]	valid_0's rmse: 8.77039	valid_0's l2: 76.9198
[550]	valid_0'

LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=5000,
              n_jobs=-1, random_state=42, subsample=0.8)

⑥ 予測（検証・テスト）
* 検証RMSE用の予測 → 提出用testの予測

In [ ]:
# 検証予測
valid_pred = lgbm.predict(X_valid_t)

# テスト予測（id除外→前処理→予測）
test_X = test.drop("id", axis=1)
test_X_t = preprocessor.transform(test_X)

test_pred = lgbm.predict(test_X_t)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


⑦ RMSEで評価
* Kaggleと同じ評価指標 RMSE を計算
* 「平均で何点ズレているか」を数値で確認

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
rmse

np.float64(8.75304122571146)

⑧ 提出用CSVを作成
* Kaggle指定フォーマットに予測結果を入れる
* 提出可能な submission.csv を作る

In [ ]:
# （参考）提出用CSVを作成し、指定フォルダに保存する

# 保存先フォルダ
submit_path = "/content/drive/MyDrive/Kaggle研究会/2026年1月提出物"

# submission 作成
submission = sample_sub.copy()
submission["exam_score"] = test_pred

# CSVとして保存
submission.to_csv(f"{submit_path}/submission20250118_02.csv", index=False)

# 確認
submission.head()

submit_dir = "/content/drive/MyDrive/Kaggle研究会/2026年1月提出物"

submission = sample_sub.copy()
submission["exam_score"] = test_pred
submission_path = f"{submit_dir}/submission20250118_03_lgbm.csv"
submission.to_csv(submission_path, index=False)

submission_path


'/content/drive/MyDrive/Kaggle研究会/2026年1月提出物/submission20250118_03_lgbm.csv'